### Importing 

In [3]:
from bertopic import BERTopic
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np  
import umap
import hdbscan
from sentence_transformers import SentenceTransformer
from google import genai
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance, OpenAI, PartOfSpeech
import spacy 
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance, OpenAI, PartOfSpeech
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary
from itertools import product
df = pd.read_csv('df_long_clean.csv')
df["Event_ID"] = df["Event"].astype("category").cat.codes

# Herorden kolommen
cols = df.columns.tolist()
cols.insert(cols.index("Event"), cols.pop(cols.index("Event_ID")))
df = df[cols]


In [4]:
docs = df["enriched_lie"].tolist()
targets = df["Event_ID"].tolist()
classes = df["Event"].tolist()
target_names = df["Event"].unique().tolist()  # unieke lijst van 8 events



In [5]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedding_model.encode(docs, show_progress_bar=True)

Batches: 100%|██████████| 94/94 [00:08<00:00, 10.82it/s]


In [6]:
# from transformers import pipeline
# from bertopic.representation import TextGeneration

# prompt = "I have a topic described by the following keywords: [KEYWORDS]. Based on the previous keywords, what is this topic about describe in max 3 words?"

# # Create your representation model
# generator = pipeline('text2text-generation', model='google/flan-t5-base')
# representation_model = TextGeneration(generator)

In [11]:
#HDBSCAN 
hdbscan_min_cluster_size = 100  # Minimum aantal punten in een cluster
# UMAP
umap_n_neighbors = 40  # Aantal buren voor UMAP
umap_n_components = 30  # Aantal dimensies voor UMAP
umap_min_dist = 0.1  # Minimum afstand tussen punten in UMAP


# CountVectorizer
vectorizer_min_df = 2
vectorizer_ngram_range = (1, 3)  # langere zinsdelen meenemen

vectorizer_stop_words = "english"  # Engelse stopwoorden gebruiken


hdbscan_model = HDBSCAN(min_cluster_size=hdbscan_min_cluster_size, 
                        metric='euclidean', 
                        cluster_selection_method='eom', 
                        prediction_data=True)


umap_model = UMAP(n_neighbors=15,
                  n_components=5, 
                    min_dist=0.0,
                  metric='cosine')


vectorizer_model = CountVectorizer(
    stop_words=vectorizer_stop_words,
    min_df=vectorizer_min_df,
    ngram_range=vectorizer_ngram_range
)


representation_model = PartOfSpeech("en_core_web_sm")


In [12]:
topic_model = BERTopic(

  # Pipeline models
  embedding_model=embedding_model
  ,
  umap_model=umap_model,
  hdbscan_model=hdbscan_model,
  vectorizer_model=vectorizer_model,
  representation_model=representation_model,

  # Hyperparameters
  top_n_words=10,
  verbose=True
)

# Train model
topics, probs = topic_model.fit_transform(docs, embeddings)

# Show topics
topic_model.get_topic_info()


2025-10-14 11:06:45,922 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-10-14 11:06:54,802 - BERTopic - Dimensionality - Completed ✓
2025-10-14 11:06:54,805 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-10-14 11:06:55,030 - BERTopic - Cluster - Completed ✓
2025-10-14 11:06:55,033 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-10-14 11:06:55,896 - BERTopic - Representation - Completed ✓


,Topic,Count,Name,Representation,Representative_Docs
0,-1,1478,-1_time_work_late_deadline,"[time, work, late, deadline, money, day, team,...","[first time ever experiencing that, I was very..."
1,0,543,0_ticket_car_bus_driving,"[ticket, car, bus, driving, driver, speed, tra...","[I always have a ticket on me, I always had a ..."
2,1,446,1_interview_job_company_position,"[interview, job, company, position, experience...","[d not even stutter during the interview, It w..."
3,2,325,2_surgery_hospital_pain_emergency,"[surgery, hospital, pain, emergency, heart, he...",[He had to conduct another surgery to remove t...
4,3,216,3_exam_test_instructions_project,"[exam, test, instructions, project, time, work...","[I studied a lot for this exam, because I had ..."


In [13]:
topic_model.visualize_documents(docs, embeddings=embeddings)

In [10]:
topics_per_class = topic_model.topics_per_class(docs, classes=classes)

topic_model.visualize_topics_per_class(topics_per_class, top_n_topics=13)


9it [00:04,  2.02it/s]


### Paramater Tuning 

In [ ]:
# param_grid = {
#     'umap_n_neighbors': [15, 20, 25, 30],
#     'umap_min_dist': [0.0, 0.1, 0.2],
#     'hdbscan_min_cluster_size': [10, 20, 25, 30, 35],
#     'hdbscan_min_samples': [5, 10, 15],
#     'vectorizer_ngram_range': [(1, 2)]
# }

# # Genereer alle combinaties
# keys, values = zip(*param_grid.items())
# param_combinations = [dict(zip(keys, v)) for v in product(*values)]

# # Maak een lege lijst om de resultaten op te slaan
# results_list = []

# for params in param_combinations:
#     # Stap 1: Initialiseer BERTopic met parameters
#     umap_model = UMAP(
#         n_neighbors=params['umap_n_neighbors'],
#         n_components=5,
#         min_dist=params['umap_min_dist'],
#         metric='cosine',
#         random_state=42
#     )
#     hdbscan_model = HDBSCAN(
#         min_cluster_size=params['hdbscan_min_cluster_size'],
#         min_samples=params['hdbscan_min_samples'],
#         metric='euclidean',
#         prediction_data=True
#     )
#     vectorizer_model = CountVectorizer(
#         ngram_range=params['vectorizer_ngram_range'],
#         stop_words="english"
#     )

#     topic_model = BERTopic(
#         umap_model=umap_model,
#         hdbscan_model=hdbscan_model,
#         vectorizer_model=vectorizer_model,
#         verbose=False
#     )

#     # Fit met vooraf berekende embeddings
#     topics, _ = topic_model.fit_transform(docs, embeddings)
    
#     # Stap 2: Gebruik BERTopic's interne tokenizer voor Gensim
#     vectorizer = topic_model.vectorizer_model
#     analyzer = vectorizer.build_analyzer()
#     tokenized_docs = [analyzer(doc) for doc in docs]
#     dictionary = Dictionary(tokenized_docs)
#     corpus = [dictionary.doc2bow(doc) for doc in tokenized_docs]
    
#     # Stap 3: Extraheer topics voor Gensim
#     topic_words = [[word for word, _ in topic_model.get_topic(topic) if word != ""]
#                    for topic in topic_model.get_topics().keys() if topic != -1]
    
#     # Check of topics bestaan voordat de coherentie wordt berekend
#     if not topic_words:
#         print(f"Skipping parameters ({params}): No topics found.")
#         continue
    
#     # Stap 4: Bereken C_v coherentie en aantal topics
#     coherence_model = CoherenceModel(
#         topics=topic_words,
#         texts=tokenized_docs,
#         dictionary=dictionary,
#         coherence='c_v'
#     )
#     coherence = coherence_model.get_coherence()
#     num_topics = len(set(topics)) - 1
    
#     print(f"Params: {params}, Coherence: {coherence:.4f}, Topics: {num_topics}")

#     # Voeg de resultaten toe aan de lijst
#     result_dict = params.copy() # Maak een kopie om de originele params niet te wijzigen
#     result_dict['coherence_score'] = coherence
#     result_dict['num_topics'] = num_topics
#     results_list.append(result_dict)

# # Converteer de lijst van dictionaries naar een DataFrame
# df_results = pd.DataFrame(results_list)

# # Sorteer en toon de resultaten
# df_results = df_results.sort_values(by='coherence_score', ascending=False)
# print("\n---")
# print("Grid Search Resultaten (gesorteerd op coherentie):")
# print(df_results)

Params: {'umap_n_neighbors': 15, 'umap_min_dist': 0.0, 'hdbscan_min_cluster_size': 10, 'hdbscan_min_samples': 5, 'vectorizer_ngram_range': (1, 2)}, Coherence: 0.4681, Topics: 73
Params: {'umap_n_neighbors': 15, 'umap_min_dist': 0.0, 'hdbscan_min_cluster_size': 10, 'hdbscan_min_samples': 10, 'vectorizer_ngram_range': (1, 2)}, Coherence: 0.4460, Topics: 50
Params: {'umap_n_neighbors': 15, 'umap_min_dist': 0.0, 'hdbscan_min_cluster_size': 10, 'hdbscan_min_samples': 15, 'vectorizer_ngram_range': (1, 2)}, Coherence: 0.4804, Topics: 37
Params: {'umap_n_neighbors': 15, 'umap_min_dist': 0.0, 'hdbscan_min_cluster_size': 20, 'hdbscan_min_samples': 5, 'vectorizer_ngram_range': (1, 2)}, Coherence: 0.4115, Topics: 31
Params: {'umap_n_neighbors': 15, 'umap_min_dist': 0.0, 'hdbscan_min_cluster_size': 20, 'hdbscan_min_samples': 10, 'vectorizer_ngram_range': (1, 2)}, Coherence: 0.3951, Topics: 27
Params: {'umap_n_neighbors': 15, 'umap_min_dist': 0.0, 'hdbscan_min_cluster_size': 20, 'hdbscan_min_samples

In [ ]:
df_results

,umap_n_neighbors,umap_min_dist,hdbscan_min_cluster_size,hdbscan_min_samples,vectorizer_ngram_range,coherence_score,num_topics
24,15,0.1,30,5,"(1, 2)",0.500425,15
12,15,0.0,35,5,"(1, 2)",0.493437,17
135,30,0.0,10,5,"(1, 2)",0.487444,61
39,15,0.2,30,5,"(1, 2)",0.482463,14
9,15,0.0,30,5,"(1, 2)",0.481005,22
...,...,...,...,...,...,...,...
14,15,0.0,35,15,"(1, 2)",0.372745,13
143,30,0.0,25,15,"(1, 2)",0.372621,12
43,15,0.2,35,10,"(1, 2)",0.367623,10
44,15,0.2,35,15,"(1, 2)",0.357953,8
